# Phase 1 : extraction multi-sources

Projet final Machine Learning, blocs 6 et 8.
Sujet : prédire quelle équipe gagne une partie professionnelle de League of Legends
à partir du seul état de jeu observable à la minute 15.

## Objectif de ce notebook

Charger les trois sources définies en phase 0, dans trois formats différents, en
maîtrisant l'encodage et les types, puis produire le tableau récapitulatif des sources.

Ce notebook ne nettoie rien et ne construit aucune feature. Il charge, il vérifie, il
documente. Le filtrage des parties incomplètes est le sujet de la phase 3, et la
construction des variables celui de la phase 4. Les problèmes repérés ici sont donc
listés en fin de notebook et renvoyés à la phase qui les traite.

## Sources et licences

| Source | Format | Origine | Accès et licence |
|---|---|---|---|
| Oracle's Elixir 2022-2026 | CSV | oracleselixir.com/tools/downloads | Libre et gratuit, attribution demandée |
| Riot Data Dragon | JSON | ddragon.leagueoflegends.com | Libre, sans clé API |
| Référentiel des ligues | XLSX | Construction manuelle | Produit par ce projet |

**Attribution.** Les données de match proviennent d'Oracle's Elixir, maintenu par
Tim Sevenhuysen (oracleselixir.com). Leur usage est libre à condition de citer la
source, ce que fait ce notebook et ce que fera la présentation de soutenance.

## 0. Configuration

Le notebook vit dans `notebooks/`, les fonctions réutilisables dans `src/`. On ajoute
la racine du projet au `sys.path` pour que les imports fonctionnent depuis un noyau
neuf, sans dépendre du répertoire courant.

Toute la configuration du projet (chemins, saisons, listes de colonnes, politique de
fuite de données) est centralisée dans `src/config.py`. Aucun nom de colonne n'est
codé en dur dans les notebooks.

In [1]:
import csv
import json
import sys
from pathlib import Path

import pandas as pd

# La racine du projet est le dossier parent de notebooks/. Le test rend la cellule
# rejouable que le noyau démarre dans notebooks/ ou à la racine.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, extraction, quality

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

print("Racine du projet       :", ROOT)
print("Données brutes         :", config.DATA_RAW)
print("Saisons attendues      :", config.SEASONS)
print("Instant de prédiction  : minute", config.PREDICTION_MINUTE)
print("Version de pandas      :", pd.__version__)

Racine du projet       : E:\LWP(LoLWinPrediciton)\lol-win-prediction
Données brutes         : E:\LWP(LoLWinPrediciton)\lol-win-prediction\data\raw
Saisons attendues      : [2022, 2023, 2024, 2025, 2026]
Instant de prédiction  : minute 15
Version de pandas      : 2.3.3


## 1. Source 1 : Oracle's Elixir (CSV)

C'est la source principale, et la seule qui contienne la cible `result`.

Structure du fichier : **douze lignes par partie**. Les dix premières décrivent les
joueurs (`position` valant top, jng, mid, bot ou sup), les deux dernières sont les
agrégats d'équipe (`position` valant `team`).

La modélisation se fait au niveau **équipe**, donc sur les deux lignes agrégées. Les
lignes joueur ne sont pas jetées pour autant : elles seules associent un champion à un
poste, ce dont la phase 4 aura besoin. On en conserve donc une projection légère.

À noter pour la phase 4 : les lignes équipe portent déjà les colonnes `pick1` à `pick5`,
qui donnent la composition sans passer par les lignes joueur. Elles sont renseignées sur
environ 80 % des lignes en 2022 et 2023, et plus de 99 % à partir de 2025. Les lignes
joueur restent donc utiles là où `pick1` à `pick5` sont vides, et pour connaître le poste
de chaque champion.

### 1.1 Disponibilité des fichiers

Oracle's Elixir est redistribué via un dossier Google Drive. Le téléchargement anonyme
est soumis à un quota journalier : au-delà, Drive renvoie une page d'erreur au lieu du
fichier. La cellule suivante tente le téléchargement automatique et, en cas d'échec,
indique la marche à suivre manuelle.

In [2]:
manquantes = extraction.missing_seasons()

if manquantes:
    print("Saisons absentes de data/raw :", manquantes)
    print("Tentative de téléchargement automatique ...")
    extraction.download_oracles_elixir(manquantes)
else:
    print("Les cinq fichiers de saison sont déjà présents dans data/raw.")

Les cinq fichiers de saison sont déjà présents dans data/raw.


In [3]:
# Garde-fou. Sans les CSV, tout le reste du notebook échouerait vingt cellules plus
# bas sur une erreur illisible. On préfère échouer ici, avec la marche à suivre.
restantes = extraction.missing_seasons()

if restantes:
    raise FileNotFoundError(
        f"Saisons absentes de data/raw : {restantes}.\n"
        "Google Drive refuse les téléchargements anonymes au-delà de son quota "
        "journalier.\n"
        "Marche à suivre : ouvrir https://oracleselixir.com/tools/downloads dans un "
        "navigateur connecté à un compte Google, télécharger les fichiers de saison "
        "à la main, puis les déposer dans data/raw/ sous leur nom d'origine, par "
        "exemple 2025_LoL_esports_match_data_from_OraclesElixir.csv"
    )

print("Les cinq fichiers de saison sont disponibles.")

Les cinq fichiers de saison sont disponibles.


### 1.2 Reconnaissance du fichier brut

Avant de laisser pandas deviner quoi que ce soit, on applique la checklist du guide de
phase 1 sur un fichier témoin : encodage, séparateur, présence d'un en-tête, lignes à
ignorer.

Le point de vigilance est l'encodage. `latin-1` décode n'importe quelle séquence
d'octets sans jamais lever d'erreur : il accepterait donc un fichier UTF-8 en le
corrompant silencieusement. On teste les encodages du plus strict au plus permissif et
on retient le premier qui passe.

In [4]:
fichier_temoin = config.DATA_RAW / extraction.OE_FILENAME.format(season=config.SEASONS[-1])
print("Fichier témoin :", fichier_temoin.name)
print("Taille         :", round(fichier_temoin.stat().st_size / 1e6, 1), "Mo")
print()

# On lit 200 lignes entières en binaire. Découper sur un saut de ligne garantit qu'on
# ne coupe pas un caractère multi-octets en deux, ce qui produirait un faux négatif.
with fichier_temoin.open("rb") as f:
    echantillon_octets = b"".join(f.readline() for _ in range(200))

print("BOM UTF-8 présent :", echantillon_octets.startswith(b"\xef\xbb\xbf"))
print()

encodage_retenu = None
for encodage in ("utf-8", "cp1252", "latin-1"):
    try:
        echantillon_octets.decode(encodage)
    except UnicodeDecodeError:
        print(f"  {encodage:<10} echec de décodage")
        continue
    print(f"  {encodage:<10} décodage réussi")
    if encodage_retenu is None:
        encodage_retenu = encodage

print()
print("Encodage retenu :", encodage_retenu)

Fichier témoin : 2026_LoL_esports_match_data_from_OraclesElixir.csv
Taille         : 67.5 Mo

BOM UTF-8 présent : False

  utf-8      décodage réussi
  cp1252     décodage réussi
  latin-1    décodage réussi

Encodage retenu : utf-8


In [5]:
echantillon = echantillon_octets.decode(encodage_retenu)

# csv.Sniffer compare la ponctuation de plusieurs lignes entières pour deviner le
# séparateur. On lui passe donc un nombre de lignes et non une tranche de caractères :
# avec 165 colonnes une seule ligne dépasse 2 000 caractères, si bien qu'une tranche
# de 5 000 caractères se terminerait au milieu d'une ligne et ferait échouer la
# détection, faute de lignes complètes à comparer.
extrait = "\n".join(echantillon.splitlines()[:20])

try:
    dialecte = csv.Sniffer().sniff(extrait)
    separateur = dialecte.delimiter
    entete_detecte = csv.Sniffer().has_header(extrait)
except csv.Error as exc:
    print("Sniffer en échec :", exc, "| on retient la virgule par défaut")
    separateur, entete_detecte = ",", True

print("Séparateur détecté :", repr(separateur))
print("En-tête détecté    :", entete_detecte)

colonnes_brutes = echantillon.splitlines()[0].lstrip("\ufeff").split(separateur)
print("Nombre de colonnes :", len(colonnes_brutes))
print()
print("Quinze premières colonnes :")
print(colonnes_brutes[:15])

Séparateur détecté : ','
En-tête détecté    : True
Nombre de colonnes : 165

Quinze premières colonnes :
['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid']


### 1.3 Comparaison des schémas de saison

Le jeu évolue et Oracle's Elixir suit : les larves du Néant arrivent en 2024, Atakhan
en 2025. On s'attend donc à ce que les cinq fichiers n'aient pas les mêmes colonnes, et
on vérifie avant de concaténer. Lire `nrows=0` ne charge que la ligne d'en-tête,
l'opération est instantanée.

In [6]:
entetes = {}
for saison in config.SEASONS:
    chemin = config.DATA_RAW / extraction.OE_FILENAME.format(season=saison)
    entetes[saison] = pd.read_csv(chemin, nrows=0, encoding=encodage_retenu)
    print(f"  saison {saison} : {entetes[saison].shape[1]} colonnes")

print()
colonnes_instables = quality.season_schema_diff(entetes)

  saison 2022 : 165 colonnes
  saison 2023 : 165 colonnes
  saison 2024 : 165 colonnes
  saison 2025 : 165 colonnes
  saison 2026 : 165 colonnes

0 columns are not present in every season:
Empty DataFrame
Columns: [2022, 2023, 2024, 2025, 2026]
Index: []


**Le résultat contredit l'attente, et c'est un vrai résultat.** Les cinq fichiers
exposent exactement les mêmes 165 colonnes. Oracle's Elixir ne fait pas croître son
schéma au fil des saisons : il republie chaque année avec le schéma courant et laisse
vides les colonnes qui n'avaient pas de sens à l'époque.

La conséquence est importante pour la suite. Comparer les en-têtes ne détecte **rien**,
et donnerait une fausse assurance à qui s'arrêterait là. La dérive est bien présente,
mais elle est dans le **taux de remplissage**, pas dans la liste des colonnes. On la
mesure donc directement.

In [7]:
# Taux de remplissage des colonnes qui dépendent de la saison, sur les lignes équipe.
# C'est ici que se voit la dérive que la comparaison d'en-têtes ne montre pas.
COLONNES_DATEES = ["void_grubs", "atakhans"]

remplissage = []
for saison in config.SEASONS:
    chemin = config.DATA_RAW / extraction.OE_FILENAME.format(season=saison)
    extrait_saison = pd.read_csv(
        chemin,
        usecols=lambda c: c in COLONNES_DATEES + ["position"],
        encoding=encodage_retenu,
        low_memory=False,
    )
    lignes_equipe_saison = extrait_saison[extrait_saison["position"] == "team"]
    remplissage.append({
        "saison": saison,
        **{f"{c}_pct_rempli": round(100 * lignes_equipe_saison[c].notna().mean(), 1)
           for c in COLONNES_DATEES},
    })

print("Taux de remplissage, en pourcentage des lignes équipe :")
print(pd.DataFrame(remplissage).to_string(index=False))

Taux de remplissage, en pourcentage des lignes équipe :
 saison  void_grubs_pct_rempli  atakhans_pct_rempli
   2022                    6.4                  0.0
   2023                    7.3                  0.0
   2024                   94.4                  0.0
   2025                  100.0                 91.6
   2026                  100.0                 82.1


La dérive est nette et correspond bien à l'histoire du jeu : les larves du Néant passent
de quasi absentes à presque toujours renseignées en 2024, Atakhan reste vide jusqu'en
2025. Ces vides sont **structurels** : la donnée n'existait pas, elle n'a pas été perdue.
On n'impute pas une colonne qui n'avait pas de sens à l'époque, et la phase 3 devra
traiter ces blocs autrement que les valeurs manquantes ordinaires.

Ces deux colonnes sont de toute façon dans `config.LEAKY_COLUMNS`, puisqu'elles
décrivent des objectifs pris après la minute 15. Elles ne seront donc pas des features.
Le point qui compte ici est la méthode : sur ce jeu de données, un contrôle de schéma
doit porter sur le remplissage, pas sur la présence des colonnes.

### 1.4 Chargement des cinq saisons

Deux précautions de typage.

**`patch` reste du texte.** La colonne vaut par exemple `"12.01"`. Oracle's Elixir
complète le numéro mineur sur deux chiffres, ce qui fait que l'ordre alphabétique est ici
correct. Mais lue comme un flottant, `"12.01"` et `"12.10"` deviendraient `12.01` et
`12.1`, et le tri les inverserait. Le zéro de tête, qui est justement ce qui sauve
l'ordre, serait perdu à la conversion. La phase 4 construira `patch_seq`, le rang du
patch dans sa saison, qui est la seule représentation qui survive au passage à 2026.

**Les identifiants restent du texte.** `gameid` et `teamid` sont des chaînes. Les
laisser en inférence produirait des flottants dès qu'une valeur manque, et `12345.0`
ne se joint pas à `12345`.

Sur la mémoire : les cinq saisons brutes tiennent difficilement en RAM simultanément.
On charge donc **une saison à la fois**, on en extrait tout de suite les lignes équipe
et une projection légère des lignes joueur, puis on libère la saison complète.

In [8]:
OE_DTYPES = {
    "gameid": "string",
    "teamid": "string",
    "playerid": "string",
    "patch": "string",
    "league": "string",
    "split": "string",
    "position": "string",
    "teamname": "string",
    "playername": "string",
    "champion": "string",
    "side": "string",
    "datacompleteness": "string",
}

# Seules ces colonnes des lignes joueur nous serviront, en phase 4, à reconstruire les
# cinq picks d'une équipe. Le reste des lignes joueur est de l'agrégat de fin de partie.
COLONNES_JOUEURS = ["gameid", "teamid", "side", "position", "playername", "champion", "year", "patch"]

blocs_equipe, blocs_joueurs, inventaire = [], [], []

for saison in config.SEASONS:
    chemin = config.DATA_RAW / extraction.OE_FILENAME.format(season=saison)

    # On restreint le dictionnaire de types aux colonnes réellement présentes dans la
    # saison : le schéma bouge d'une année à l'autre, comme la section 1.3 vient de le
    # montrer.
    dtypes_saison = {c: t for c, t in OE_DTYPES.items() if c in entetes[saison].columns}
    df = pd.read_csv(chemin, dtype=dtypes_saison, encoding=encodage_retenu, low_memory=False)

    equipes = df[df["position"] == "team"]
    joueurs = df[df["position"] != "team"]

    inventaire.append({
        "saison": saison,
        "taille_mo": round(chemin.stat().st_size / 1e6, 1),
        "lignes_totales": len(df),
        "colonnes": df.shape[1],
        "parties": df["gameid"].nunique(),
        "lignes_equipe": len(equipes),
        "lignes_joueur": len(joueurs),
    })

    blocs_equipe.append(equipes)
    blocs_joueurs.append(joueurs[COLONNES_JOUEURS])

    print(f"  saison {saison} chargée : {len(df):>7,} lignes, {df.shape[1]} colonnes")
    del df, equipes, joueurs

inventaire = pd.DataFrame(inventaire)
print()
print(inventaire.to_string(index=False))

  saison 2022 chargée : 150,348 lignes, 165 colonnes


  saison 2023 chargée : 133,272 lignes, 165 colonnes


  saison 2024 chargée : 122,340 lignes, 165 colonnes


  saison 2025 chargée : 120,492 lignes, 165 colonnes


  saison 2026 chargée : 100,812 lignes, 165 colonnes

 saison  taille_mo  lignes_totales  colonnes  parties  lignes_equipe  lignes_joueur
   2022       97.6          150348       165    12529          25058         125290
   2023       85.9          133272       165    11106          22212         111060
   2024       79.1          122340       165    10195          20390         101950
   2025       79.2          120492       165    10041          20082         100410
   2026       67.5          100812       165     8401          16802          84010


In [9]:
# pd.concat aligne les colonnes par leur nom. Les colonnes absentes d'une saison sont
# remplies par NaN : ce sont exactement les blocs structurels repérés en 1.3.
oe = pd.concat(blocs_equipe, ignore_index=True)
oe_joueurs = pd.concat(blocs_joueurs, ignore_index=True)
del blocs_equipe, blocs_joueurs

print(f"Lignes équipe : {len(oe):>8,} sur {oe.shape[1]} colonnes")
print(f"Lignes joueur : {len(oe_joueurs):>8,} sur {oe_joueurs.shape[1]} colonnes (projection)")
print(f"Parties       : {oe['gameid'].nunique():>8,}")

Lignes équipe :  104,544 sur 165 colonnes
Lignes joueur :  522,720 sur 8 colonnes (projection)
Parties       :   52,272


### 1.5 Dates et types

Le guide met en garde contre les dates laissées en texte. On les convertit avec un
**format explicite** plutôt qu'en laissant pandas inférer, car l'inférence peut changer
d'interprétation d'un bloc à l'autre et confondre jour et mois sans rien signaler.

On vérifie ensuite que le même format couvre bien les cinq saisons. Un changement de
format en cours de route est précisément le genre de détail qui casse un tri
chronologique, dont dépend tout le projet.

In [10]:
FORMATS_DATE = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d"]

def parser_dates(serie, formats=FORMATS_DATE):
    # Essaie chaque format explicite et retient le premier qui ne produit aucun échec.
    # Un échec est une valeur non nulle au départ que le format n'a pas su lire.
    # Renvoie le triplet (serie_convertie, format_utilise, nombre_echecs).
    meilleur = (None, None, len(serie) + 1)
    for fmt in formats:
        convertie = pd.to_datetime(serie, format=fmt, errors="coerce")
        echecs = int((convertie.isna() & serie.notna()).sum())
        if echecs == 0:
            return convertie, fmt, 0
        if echecs < meilleur[2]:
            meilleur = (convertie, fmt, echecs)
    return meilleur

print("Format de date retenu, saison par saison :")
for saison in config.SEASONS:
    masque = oe["year"] == saison
    _, fmt, echecs = parser_dates(oe.loc[masque, "date"])
    print(f"  {saison} : {fmt!r:<22} {echecs} échec(s) sur {int(masque.sum()):,} lignes")

Format de date retenu, saison par saison :
  2022 : '%Y-%m-%d %H:%M:%S'    0 échec(s) sur 24,366 lignes
  2023 : '%Y-%m-%d %H:%M:%S'    0 échec(s) sur 22,188 lignes
  2024 : '%Y-%m-%d %H:%M:%S'    0 échec(s) sur 20,380 lignes
  2025 : '%Y-%m-%d %H:%M:%S'    0 échec(s) sur 20,312 lignes
  2026 : '%Y-%m-%d %H:%M:%S'    0 échec(s) sur 17,288 lignes


In [11]:
oe["date"], format_date, echecs_date = parser_dates(oe["date"])

if echecs_date:
    raise ValueError(
        f"{echecs_date} dates non converties avec le format {format_date!r}. "
        "Ajouter le format manquant à FORMATS_DATE."
    )

print("Format appliqué  :", format_date)
print("Période couverte :", oe["date"].min(), "->", oe["date"].max())
print("Type de la colonne date :", oe["date"].dtype)

Format appliqué  :

 %Y-%m-%d %H:%M:%S
Période couverte : 2022-01-10 07:44:08 -> 2026-09-06 22:34:24
Type de la colonne date : datetime64[ns]


In [12]:
# Contrôle des types sur les colonnes qui comptent pour la suite du projet.
colonnes_temoins = ["gameid", "teamid", "date", "patch", "league", "side", "result"] + config.REQUIRED_AT15

apercu_types = pd.DataFrame({
    "type": oe[colonnes_temoins].dtypes.astype(str),
    "exemple": [oe[c].dropna().iloc[0] if oe[c].notna().any() else None for c in colonnes_temoins],
})
print(apercu_types.to_string())

print()
print("Vérification du piège du patch")
print("  type de patch :", oe["patch"].dtype)
print("  exemples      :", sorted(oe["patch"].dropna().unique())[:8])

                        type                                  exemple
gameid                string                    ESPORTSTMNT01_2690210
teamid                string  oe:team:fa3d687a87bcae80362f784a7da571d
date          datetime64[ns]                      2022-01-10 07:44:08
patch                 string                                    12.01
league                string                                     LCKC
side                  string                                     Blue
result                 int64                                        0
goldat15             float64                                  24806.0
xpat15               float64                                  28001.0
csat15               float64                                    487.0
golddiffat15         float64                                    107.0
xpdiffat15           float64                                  -1617.0

Vérification du piège du patch
  type de patch : string
  exemples      : ['12.01', '12.0

### 1.6 Vérifications post-chargement

On applique la checklist de validation du guide : dimensions, unicité des identifiants,
cohérence de la structure, aperçu de la cible.

La règle d'inclusion des parties, elle, n'est pas appliquée ici. Mesurer la complétude
est le livrable de la phase 2, et supprimer des lignes celui de la phase 3. On se
contente donc de compter, sans filtrer.

In [13]:
print("Structure")
print("  parties distinctes :", f"{oe['gameid'].nunique():,}")
print("  lignes équipe      :", f"{len(oe):,}")
print()

print("Nombre de lignes équipe par partie (2 attendu)")
print(oe.groupby("gameid").size().value_counts().sort_index().to_string())
print()

print("Équilibre de la cible result")
print(oe["result"].value_counts(dropna=False, normalize=True).round(4).to_string())

Structure
  parties distinctes : 52,272
  lignes équipe      : 104,544

Nombre de lignes équipe par partie (2 attendu)
2    52272

Équilibre de la cible result
result
0    0.5
1    0.5


Le cadrage s'appuie sur une propriété forte : chaque partie produit exactement une ligne
gagnante et une ligne perdante, donc une cible à 50/50 par construction. Une propriété
aussi commode mérite d'être vérifiée plutôt que supposée, d'autant qu'elle justifie de ne
prévoir aucun rééquilibrage de classes.

In [14]:
vainqueurs_par_partie = oe.groupby("gameid")["result"].sum()
parties_sans_vainqueur = int((vainqueurs_par_partie != 1).sum())

print("Somme de result par partie (1 attendu) :")
print(vainqueurs_par_partie.value_counts().sort_index().to_string())
print()
print("Moyenne de result :", round(oe["result"].mean(), 6))
print("Parties sans vainqueur unique :", parties_sans_vainqueur)

if parties_sans_vainqueur:
    anormales = vainqueurs_par_partie[vainqueurs_par_partie != 1].index
    print()
    print(oe[oe["gameid"].isin(anormales)][
        ["gameid", "league", "year", "teamname", "result"]
    ].to_string(index=False))

Somme de result par partie (1 attendu) :
result
0        3
1    52269

Moyenne de result : 0.499971
Parties sans vainqueur unique : 3

               gameid league  year            teamname  result
ESPORTSTMNT01_3408461    LCK  2023        Liiv SANDBOX       0
ESPORTSTMNT01_3408461    LCK  2023                  T1       0
      LOLTMNT01_52116  ESLOL  2024      A One Man Army       0
      LOLTMNT01_52116  ESLOL  2024    Once Upon A Team       0
   10867-10867_game_1    LDL  2024 Ultra Prime Academy       0
   10867-10867_game_1    LDL  2024   Oh My God Academy       0


La propriété est presque vraie, et l'écart est instructif. Trois parties portent `result`
à 0 sur leurs **deux** lignes : aucune équipe n'y est déclarée gagnante. Ce sont des
parties annulées ou rejouées dont le résultat n'a jamais été consolidé.

L'effet sur l'équilibre est infime, mais ces six lignes sont inutilisables en
apprentissage supervisé, puisque leur cible ne décrit aucun résultat réel. La phase 3 les
supprimera par paires, exactement comme les parties au snapshot incomplet.

Le contrôle d'unicité demande une précaution. `duplicated` considère deux `NaN` comme
égaux, alors que `teamid` est parfois absent. Deux équipes différentes d'une même partie,
toutes deux sans `teamid`, ressembleraient donc à un doublon sans en être un.

On teste donc l'unicité **sur les seules lignes où la clé est renseignée**, et on compte
séparément les `teamid` manquants. Sans cette séparation, le contrôle échouerait pour la
mauvaise raison et enverrait la phase 3 chercher un problème qui n'existe pas.

In [15]:
teamid_manquant = int(oe["teamid"].isna().sum())
cle_renseignee = oe[oe["teamid"].notna()]
doublons = int(cle_renseignee.duplicated(subset=["gameid", "teamid"]).sum())

print(f"Lignes équipe sans teamid          : {teamid_manquant:,} "
      f"({100 * teamid_manquant / len(oe):.2f} %)")
print(f"Doublons gameid + teamid, clé renseignée : {doublons}")
print()

# Contre-preuve : les lignes sans teamid opposent-elles bien deux équipes distinctes ?
sans_id = oe[oe["teamid"].isna()]
if len(sans_id):
    equipes_par_partie = sans_id.groupby("gameid")["teamname"].nunique()
    print("Parties concernées par un teamid manquant :", f"{len(equipes_par_partie):,}")
    print("Noms d'équipe distincts par partie :")
    print(equipes_par_partie.value_counts().sort_index().to_string())
    print()
    print("Ligues concernées :")
    print(sans_id["league"].value_counts().head(5).to_string())

Lignes équipe sans teamid          : 1,800 (1.72 %)
Doublons gameid + teamid, clé renseignée : 0

Parties concernées par un teamid manquant : 1,475
Noms d'équipe distincts par partie :
teamname
1    1150
2     325

Ligues concernées :
league
LJL      392
LPLOL    220
ESLOL    205
CD       134
PRMP      90


### 1.7 Anomalies sur les colonnes de découpage

Le split du projet repose sur `year` : entraînement de 2022 à 2025, test sur 2026. Toute
valeur inattendue dans cette colonne enverrait des lignes hors des deux jeux, en silence.
On la contrôle donc explicitement, ainsi que la cohérence entre `year` et `date`.

In [16]:
print("Valeurs de year rencontrées :")
print(oe["year"].value_counts().sort_index().to_string())
print()

annees_inattendues = sorted(set(oe["year"].dropna().unique()) - set(config.SEASONS))
print("Valeurs hors des saisons attendues :", annees_inattendues)

if annees_inattendues:
    hors = oe[oe["year"].isin(annees_inattendues)]
    print(f"  {len(hors)} lignes concernées")
    print()
    print(hors[["gameid", "league", "year", "date", "patch", "teamname"]].head(6).to_string(index=False))
    print()
    # year et date racontent-ils la même histoire ?
    print("Année réelle de la date, pour ces lignes :")
    print(hors["date"].dt.year.value_counts().to_string())

Valeurs de year rencontrées :
year
2022    24366
2023    22188
2024    20380
2025    20312
2026    17288
2027       10

Valeurs hors des saisons attendues : [2027]
  10 lignes concernées

          gameid league  year                date patch        teamname
LOLTMNT04_188212    EBL  2027 2026-09-02 16:19:53 16.17     RLX Esports
LOLTMNT04_188212    EBL  2027 2026-09-02 16:19:53 16.17 Only The Family
LOLTMNT04_188222    EBL  2027 2026-09-02 17:06:11 16.17     RLX Esports
LOLTMNT04_188222    EBL  2027 2026-09-02 17:06:11 16.17 Only The Family
LOLTMNT04_188231    EBL  2027 2026-09-02 18:02:05 16.17     RLX Esports
LOLTMNT04_188231    EBL  2027 2026-09-02 18:02:05 16.17 Only The Family

Année réelle de la date, pour ces lignes :
date
2026    10


Ces lignes sont datées de septembre 2026 et jouées sur le patch 16.17, mais étiquetées
`year = 2027`. Il s'agit du début d'une saison 2027 déjà ouverte dans le calendrier de la
ligue, pas d'une erreur de date.

Le volume est négligeable, mais la décision ne l'est pas : découper le jeu sur `year`
placerait ces lignes hors du train comme du test, alors que découper sur `date` les
enverrait dans le test 2026. La phase 3 doit trancher explicitement plutôt que laisser le
hasard décider. Le constat est posé ici, l'arbitrage appartient à la phase 3.

In [17]:
# Complétude du snapshot à 15 minutes. Simple mesure, aucun filtrage à ce stade.
at15_complet = oe[config.REQUIRED_AT15].notna().all(axis=1)
print(f"Lignes équipe avec un snapshot à 15 minutes complet : "
      f"{int(at15_complet.sum()):,} sur {len(oe):,} ({100 * at15_complet.mean():.1f} %)")
print()
print("Valeurs manquantes sur les colonnes requises :")
print(oe[config.REQUIRED_AT15].isna().sum().to_string())
print()
print("Répartition du drapeau datacompleteness :")
print(oe["datacompleteness"].value_counts(dropna=False).to_string())

Lignes équipe avec un snapshot à 15 minutes complet : 92,616 sur 104,544 (88.6 %)

Valeurs manquantes sur les colonnes requises :
goldat15        11928
xpat15          11928
csat15          11928
golddiffat15    11928
xpdiffat15      11928

Répartition du drapeau datacompleteness :
datacompleteness
complete    92646
partial     11898


In [18]:
oe.head(3)

,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,participantid,side,position,playername,playerid,teamname,teamid,firstPick,champion,ban1,ban2,ban3,ban4,ban5,pick1,pick2,pick3,pick4,pick5,gamelength,...,goldat20,xpat20,csat20,opp_goldat20,opp_xpat20,opp_csat20,golddiffat20,xpdiffat20,csdiffat20,killsat20,assistsat20,deathsat20,opp_killsat20,opp_assistsat20,opp_deathsat20,goldat25,xpat25,csat25,opp_goldat25,opp_xpat25,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
0,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,100,Blue,team,<NA>,<NA>,HANJIN BRION Challengers,oe:team:fa3d687a87bcae80362f784a7da571d,1.0,<NA>,Karma,Caitlyn,Syndra,Thresh,Lulu,Renekton,Samira,Xin Zhao,LeBlanc,Leona,1713,...,31962.0,36874.0,631.0,32906.0,41821.0,715.0,-944.0,-4947.0,-84.0,5.0,10.0,7.0,7.0,22.0,5.0,40224.0,45960.0,767.0,40136.0,49931.0,864.0,88.0,-3971.0,-97.0,6.0,12.0,7.0,7.0,22.0,6.0
1,ESPORTSTMNT01_2690210,complete,NaN,LCKC,2022,Spring,0,2022-01-10 07:44:08,1,12.01,200,Red,team,<NA>,<NA>,Nongshim Esports Academy,oe:team:7c64febcd5ccff13dcd035dc6867a00,0.0,<NA>,Lee Sin,Twisted Fate,Zoe,Nautilus,Rell,Jinx,Viego,Gragas,Viktor,Alistar,1713,...,32906.0,41821.0,715.0,31962.0,36874.0,631.0,944.0,4947.0,84.0,7.0,22.0,5.0,5.0,10.0,7.0,40136.0,49931.0,864.0,40224.0,45960.0,767.0,-88.0,3971.0,97.0,7.0,22.0,6.0,6.0,12.0,7.0
2,ESPORTSTMNT01_2690219,complete,NaN,LCKC,2022,Spring,0,2022-01-10 08:38:24,1,12.01,100,Blue,team,<NA>,<NA>,T1 Esports Academy,oe:team:731b7a9fd004cdbe2bcb3da795bce47,1.0,<NA>,Sona,Jarvan IV,Caitlyn,Lulu,Lucian,Lee Sin,Jhin,Gragas,Rakan,Orianna,2114,...,31228.0,38596.0,710.0,36368.0,42069.0,758.0,-5140.0,-3473.0,-48.0,1.0,1.0,5.0,5.0,6.0,1.0,39335.0,49409.0,895.0,46615.0,57155.0,928.0,-7280.0,-7746.0,-33.0,1.0,1.0,8.0,8.0,13.0,1.0


## 2. Source 2 : Riot Data Dragon (JSON)

Data Dragon est le référentiel statique publié par Riot. Il donne, pour chaque
champion, ses `tags` (Fighter, Mage, Marksman, Tank, Assassin, Support), sa ressource
et ses statistiques de base.

Son rôle dans le projet est précis : traduire les cinq champions bruts d'une équipe en
features de composition exploitables, ce que fera la phase 4.

Accès libre, sans clé API et sans authentification. L'API est versionnée :
`versions.json` renvoie la liste des versions de la plus récente à la plus ancienne,
donc l'élément d'indice 0 est le patch courant.

### 2.1 Structure du JSON

Checklist du guide pour une source JSON : objet ou liste, clé contenant les données
utiles, niveaux d'imbrication, encodage des caractères spéciaux.

In [19]:
if not (config.DATA_RAW / "champions_en.json").exists():
    extraction.download_champions()

chemin_json = config.DATA_RAW / "champions.json"
payload = json.loads(chemin_json.read_text(encoding="utf-8"))

print("Clés de premier niveau :", list(payload.keys()))
print("type    :", payload["type"])
print("version :", payload["version"])
print("Nombre de champions :", len(payload["data"]))
print()

# `data` est un objet indexé par la clé interne du champion, pas une liste.
premiere_cle = next(iter(payload["data"]))
exemple = payload["data"][premiere_cle]
print("Clé interne d'exemple :", premiere_cle)
print("Champs d'un champion  :", list(exemple.keys()))
print()
print("  name    :", exemple["name"])
print("  tags    :", exemple["tags"])
print("  partype :", exemple["partype"])
print("  info    :", exemple["info"], "  <- niveau imbriqué à aplatir")

Clés de premier niveau : ['type', 'format', 'version', 'data']
type    : champion
version : 16.17.1
Nombre de champions : 173

Clé interne d'exemple : Aatrox
Champs d'un champion  : ['version', 'id', 'key', 'name', 'title', 'blurb', 'info', 'image', 'tags', 'partype', 'stats']

  name    : Aatrox
  tags    : ['Fighter']
  partype : Puits de sang
  info    : {'attack': 8, 'defense': 4, 'magic': 3, 'difficulty': 4}   <- niveau imbriqué à aplatir


### 2.2 Aplatissement et piège du nom de champion

Un JSON imbriqué se transforme en table en choisissant explicitement les champs à
garder. `src/extraction.py::load_champions` fait ce travail.

**Le piège, et c'est le vrai problème détecté sur cette source.** Data Dragon est
localisé. La version `fr_FR` traduit le nom d'affichage, alors qu'Oracle's Elixir écrit
les noms en anglais. Joindre sur le nom français ferait disparaître silencieusement les
champions dont la traduction diffère, sans lever la moindre erreur : la ligne
deviendrait simplement `NaN` après la jointure, et la composition de l'équipe serait
comptée avec un champion de moins.

La correction retenue : télécharger les deux locales, joindre sur le nom anglais et
conserver le nom français pour l'affichage des figures et des rapports, qui doivent
être en français. La cellule suivante quantifie l'écart.

In [20]:
champions = extraction.load_champions()
print()
print("Dimensions :", champions.shape)
print()
print(champions.dtypes.to_string())

Loaded 173 champions, tags: ['Assassin', 'Fighter', 'Mage', 'Marksman', 'Support', 'Tank']
5 champions have a French name different from the English join key

Dimensions : (173, 10)

champion_key      object
champion          object
nom_fr            object
tag_principal     object
tag_secondaire    object
partype           object
attack             int64
defense            int64
magic              int64
difficulty         int64


In [21]:
ecarts_noms = champions.loc[champions["champion"] != champions["nom_fr"],
                            ["champion_key", "champion", "nom_fr"]]

print(f"{len(ecarts_noms)} champions dont le nom français diffère du nom anglais.")
print("Ce sont autant de jointures qui auraient échoué en silence.")
ecarts_noms

5 champions dont le nom français diffère du nom anglais.
Ce sont autant de jointures qui auraient échoué en silence.


,champion_key,champion,nom_fr
55,KSante,K'Sante,K'Santé
81,MasterYi,Master Yi,Maître Yi
95,Nunu,Nunu & Willump,Nunu et Willump
117,Seraphine,Seraphine,Séraphine
171,Zoe,Zoe,Zoé


In [22]:
print("Répartition des tags principaux :")
print(champions["tag_principal"].value_counts().to_string())
print()
print("Champions sans tag secondaire :", int(champions["tag_secondaire"].isna().sum()))
champions.head(3)

Répartition des tags principaux :
tag_principal
Fighter     50
Mage        35
Marksman    29
Tank        24
Support     18
Assassin    17

Champions sans tag secondaire : 43


,champion_key,champion,nom_fr,tag_principal,tag_secondaire,partype,attack,defense,magic,difficulty
0,Aatrox,Aatrox,Aatrox,Fighter,None,Puits de sang,8,4,3,4
1,Ahri,Ahri,Ahri,Mage,Assassin,Mana,3,4,8,5
2,Akali,Akali,Akali,Assassin,None,Énergie,5,3,8,7


## 3. Source 3 : référentiel des ligues (XLSX)

Troisième source et troisième format, comme demandé par la phase 1.

Ce fichier n'est pas téléchargé : il est construit à la main, à partir de la
connaissance du circuit compétitif, par `src/extraction.py::build_league_reference`.
Il associe à chaque code de ligue son tier, sa région et son statut franchisé.

Son rôle est double. Il répond à la question business 5, sur la conversion des
avantages précoces selon le niveau de la ligue. Et surtout il fournit `region` et
`tier_ligue`, qui remplacent `league` dans les features : le circuit a été réorganisé
en 2025, la LCS devenant la LTA et le PCS fusionnant dans la LCP, si bien que les codes
de ligue ne survivent pas à la frontière entre l'entraînement et le test.

Le référentiel couvre les 84 codes de ligue observés dans les données, plus `LCK CL`
conservé comme variante d'écriture de `LCKC`. Le tier décrit un niveau de compétition,
pas une opinion sur la qualité de jeu :

| Tier | Définition |
|---|---|
| 1 | Circuit qualifiant directement pour les Worlds, et compétitions internationales |
| 2 | Ligue nationale ou régionale senior |
| 3 | Academy, challenger, development, universitaire, ou coupe secondaire |

Une colonne `confiance` a été ajoutée aux quatre prévues au cadrage. Une trentaine de
codes ne sont pas identifiables de mémoire, et les inventer aurait produit un tableau
propre mais faux. Ils ont donc été résolus en lisant les **noms d'équipes** réellement
présents dans chaque ligue, ce qui est une preuve tirée des données et non un souvenir.

L'exemple le plus net est `LAS`. Les initiales évoquent l'Amérique latine, mais ses
rosters sont T1 Esports Academy Rookies et DRX Academy : c'est la ligue academy
coréenne, et 3 026 lignes auraient été classées dans la mauvaise région.

`confiance` vaut `haute` quand la compétition est identifiée avec certitude, et
`moyenne` quand la région est établie par les rosters mais que le nom exact de la
compétition reste déduit. Pour ces lignes, c'est le tier qui est le plus incertain, pas
la région.

Checklist du guide pour une source Excel : lister les feuilles, repérer les lignes
d'en-tête à ignorer, les cellules fusionnées et les colonnes calculées.

In [23]:
chemin_xlsx = config.DATA_RAW / "referentiel_ligues.xlsx"
if not chemin_xlsx.exists():
    extraction.build_league_reference()

classeur = pd.ExcelFile(chemin_xlsx)
print("Feuilles du classeur :", classeur.sheet_names)

ligues = pd.read_excel(chemin_xlsx, sheet_name="ligues")

print("Dimensions :", ligues.shape)
print()
print(ligues.dtypes.to_string())

Feuilles du classeur : ['ligues']
Dimensions : (85, 5)

league        object
tier_ligue     int64
region        object
franchisee      bool
confiance     object


Le fichier étant produit par le projet lui-même, les pièges habituels du format Excel
sont écartés par construction : une seule feuille, en-tête en première ligne, aucune
cellule fusionnée, aucune formule, donc aucune valeur calculée à figer. Les types sont
lus correctement, `tier_ligue` en entier et `franchisee` en booléen.

La contrepartie est que sa **couverture** n'est garantie par personne. C'est le point
vérifié en section 4.

In [24]:
print("Répartition des ligues du référentiel par tier :")
print(ligues["tier_ligue"].value_counts().sort_index().to_string())
print()
print("Répartition par région :")
print(ligues["region"].value_counts().to_string())
print()
print("Niveau de confiance de la classification :")
print(ligues["confiance"].value_counts().to_string())

ligues.head(10)

Répartition des ligues du référentiel par tier :
tier_ligue
1    13
2    30
3    42

Répartition par région :
region
Europe            34
Ameriques         23
Asie-Pacifique     9
Coree              6
International      5
Chine              3
Turquie            3
CEI                1
Moyen-Orient       1

Niveau de confiance de la classification :
confiance
haute      56
moyenne    29


,league,tier_ligue,region,franchisee,confiance
0,LCK,1,Coree,True,haute
1,LPL,1,Chine,True,haute
2,LEC,1,Europe,True,haute
3,LCS,1,Ameriques,True,haute
4,LTA,1,Ameriques,True,haute
5,LTA N,1,Ameriques,True,haute
6,LTA S,1,Ameriques,True,haute
7,LCP,1,Asie-Pacifique,True,haute
8,WLDs,1,International,False,haute
9,MSI,1,International,False,haute


## 4. Clés de jointure et couverture

Le guide demande de documenter les clés qui relieront les sources. Ce projet en compte
trois.

| Lien | Clé | Granularité |
|---|---|---|
| Identité d'une ligne à modéliser | `gameid` + `teamid` | Une équipe dans une partie |
| Oracle's Elixir vers référentiel | `league` | Une ligne équipe vers une ligue |
| Oracle's Elixir vers Data Dragon | `champion` (nom anglais) | Une ligne joueur vers un champion |

Les granularités diffèrent, ce qui est le point d'attention : Oracle's Elixir est au
grain de la ligne équipe, Data Dragon au grain du champion, et le référentiel au grain
de la ligue. Les deux sources d'enrichissement se joignent donc en un vers plusieurs
sur la source principale.

Une jointure non couverte ne lève aucune erreur, elle produit des `NaN`. On mesure donc
la couverture avant de faire confiance à la jointure.

In [25]:
couverture_ligues = extraction.audit_league_coverage(oe, ligues)

Every league code is covered by the reference table.


La couverture est complète, mais l'absence de code manquant ne suffit pas. Une jointure
un vers plusieurs peut aussi **dupliquer** des lignes si la clé du référentiel n'est pas
unique : une ligue présente deux fois y multiplierait par deux les lignes équipe
correspondantes, sans le moindre message d'erreur.

On vérifie donc les deux propriétés qui comptent vraiment : le nombre de lignes est
inchangé après la jointure, et aucune valeur n'est manquante sur les deux colonnes qui
deviendront des features.

In [26]:
print("Clé du référentiel unique :", not ligues["league"].duplicated().any())

controle_jointure = oe.merge(ligues, on="league", how="left", validate="many_to_one")

print(f"Lignes avant jointure : {len(oe):,}")
print(f"Lignes après jointure : {len(controle_jointure):,}")
print()
print("Valeurs manquantes sur les colonnes issues du référentiel :")
print(controle_jointure[["region", "tier_ligue"]].isna().sum().to_string())
print()
print("Répartition des lignes équipe par tier :")
print(controle_jointure["tier_ligue"].value_counts().sort_index().to_string())
print()
print("Répartition des lignes équipe par région :")
print(controle_jointure["region"].value_counts().to_string())
print()
part_moyenne = 100 * controle_jointure["confiance"].eq("moyenne").mean()
print(f"Part des lignes classées avec une confiance moyenne : {part_moyenne:.1f} %")

Clé du référentiel unique : True


Lignes avant jointure : 104,544
Lignes après jointure : 104,544

Valeurs manquantes sur les colonnes issues du référentiel :
region        0
tier_ligue    0

Répartition des lignes équipe par tier :
tier_ligue
1    22080
2    43362
3    39102

Répartition des lignes équipe par région :
region
Europe            38190
Ameriques         21726
Coree             13142
Chine             12560
Asie-Pacifique    11696
Turquie            2680
International      2622
Moyen-Orient       1896
CEI                  32

Part des lignes classées avec une confiance moyenne : 16.5 %


`validate="many_to_one"` fait échouer la cellule si le référentiel contenait un doublon,
plutôt que de laisser passer une duplication silencieuse. Le compte de lignes est
identique avant et après, et aucune valeur n'est manquante.

Deux points à retenir pour la suite. La région `CEI` ne compte que 32 lignes, toutes en
2022 : c'est une modalité rare, que la phase 4 devra sans doute regrouper plutôt que
d'encoder telle quelle. Et environ 16 % des lignes reposent sur une classification de
confiance moyenne, ce qui est le chiffre à citer si la question business 5, sur les
ligues majeures contre les ligues mineures, donne un résultat serré.

In [27]:
noms_oe = set(oe_joueurs["champion"].dropna().unique())
noms_dd = set(champions["champion"])

absents = sorted(noms_oe - noms_dd)
print(f"Champions présents dans Oracle's Elixir mais absents de Data Dragon : {len(absents)}")
if absents:
    print(absents)
print()
print(f"Couverture des picks : {len(noms_oe & noms_dd)} champions sur {len(noms_oe)} rencontrés.")

Champions présents dans Oracle's Elixir mais absents de Data Dragon : 0

Couverture des picks : 173 champions sur 173 rencontrés.


## 5. Tableau récapitulatif des sources

Le livrable central de la phase 1, demandé en section 3 du guide.

In [28]:
recap = pd.DataFrame([
    {
        "source": "Oracle's Elixir",
        "format": "CSV",
        "fichier": "data/raw/<annee>_LoL_esports_match_data_from_OraclesElixir.csv",
        "lignes": len(oe),
        "colonnes": oe.shape[1],
        "cle_primaire": "gameid + teamid",
        "cle_jointure": "league, et champion via les lignes joueur",
        "periode": f"{oe['date'].min():%Y-%m-%d} a {oe['date'].max():%Y-%m-%d}",
        "observations": "Lignes equipe uniquement. Majorite de colonnes post-partie, a retirer en phase 3",
    },
    {
        "source": "Riot Data Dragon",
        "format": "JSON",
        "fichier": "data/raw/champions.json et champions_en.json",
        "lignes": len(champions),
        "colonnes": champions.shape[1],
        "cle_primaire": "champion_key",
        "cle_jointure": "champion, nom anglais",
        "periode": f"patch {payload['version']}, etat courant du jeu",
        "observations": "Source localisee. 5 noms differents entre fr_FR et en_US, jointure sur en_US",
    },
    {
        "source": "Referentiel des ligues",
        "format": "XLSX",
        "fichier": "data/raw/referentiel_ligues.xlsx",
        "lignes": len(ligues),
        "colonnes": ligues.shape[1],
        "cle_primaire": "league",
        "cle_jointure": "league",
        "periode": "sans objet",
        "observations": "Construit a la main. Couvre les 84 codes observes, 16 pct des lignes en confiance moyenne",
    },
])

recap.set_index("source")

,format,fichier,lignes,colonnes,cle_primaire,cle_jointure,periode,observations
source,,,,,,,,
Oracle's Elixir,CSV,data/raw/<annee>_LoL_esports_match_data_from_O...,104544,165,gameid + teamid,"league, et champion via les lignes joueur",2022-01-10 a 2026-09-06,Lignes equipe uniquement. Majorite de colonnes...
Riot Data Dragon,JSON,data/raw/champions.json et champions_en.json,173,10,champion_key,"champion, nom anglais","patch 16.17.1, etat courant du jeu",Source localisee. 5 noms differents entre fr_F...
Referentiel des ligues,XLSX,data/raw/referentiel_ligues.xlsx,85,5,league,league,sans objet,Construit a la main. Couvre les 84 codes obser...


In [29]:
print("Volume total chargé")
print(f"  lignes équipe        : {len(oe):>9,}")
print(f"  lignes joueur        : {len(oe_joueurs):>9,}")
print(f"  parties              : {oe['gameid'].nunique():>9,}")
print(f"  champions référencés : {len(champions):>9,}")
print(f"  ligues référencées   : {len(ligues):>9,}")
print()
print("Répartition des lignes équipe par saison :")
print(oe["year"].value_counts().sort_index().to_string())

Volume total chargé
  lignes équipe        :   104,544
  lignes joueur        :   522,720
  parties              :    52,272
  champions référencés :       173
  ligues référencées   :        85

Répartition des lignes équipe par saison :
year
2022    24366
2023    22188
2024    20380
2025    20312
2026    17288
2027       10


## 6. Checklist de validation de la phase 1

Plutôt qu'une liste cochée à la main, on vérifie les critères par du code. Une case
cochée à tort est une erreur qui se propage. Un test qui échoue se voit.

Un contrôle ressort en échec, et c'est voulu : les trois parties sans vainqueur sont un
défaut réel des données, pas du chargement. Le laisser en rouge ici, et le traiter en
phase 3, vaut mieux que d'assouplir le seuil jusqu'à ce que tout passe au vert.

In [30]:
controles = [
    ("Les trois sources sont chargées",
     len(oe) > 0 and len(champions) > 0 and len(ligues) > 0),
    ("Trois formats différents (CSV, JSON, XLSX)",
     True),
    ("Volume largement supérieur à 10 000 lignes",
     len(oe) > 10_000),
    ("Encodage identifié et appliqué explicitement",
     encodage_retenu is not None),
    ("Séparateur identifié",
     separateur == ","),
    ("patch conservé en texte et non en flottant",
     pd.api.types.is_string_dtype(oe["patch"])),
    ("date convertie avec un format explicite",
     pd.api.types.is_datetime64_any_dtype(oe["date"]) and echecs_date == 0),
    ("Clé gameid + teamid unique là où elle est renseignée",
     doublons == 0),
    ("Deux lignes équipe par partie, sans exception",
     oe.groupby("gameid").size().eq(2).all()),
    ("Cible result présente et binaire",
     set(oe["result"].dropna().unique()) <= {0, 1}),
    ("Une seule équipe gagnante par partie",
     parties_sans_vainqueur == 0),
    ("Schémas de saison comparés avant concaténation",
     colonnes_instables is not None),
    ("Dérive de remplissage mesurée saison par saison",
     len(remplissage) == len(config.SEASONS)),
    ("Couverture des jointures mesurée",
     couverture_ligues is not None),
    ("Tous les codes de ligue couverts par le référentiel",
     len(couverture_ligues) == 0),
    ("Jointure du référentiel sans duplication de ligne",
     len(controle_jointure) == len(oe)),
    ("Aucune valeur manquante sur region et tier_ligue",
     int(controle_jointure[["region", "tier_ligue"]].isna().sum().sum()) == 0),
    ("Tous les champions d'Oracle's Elixir résolus dans Data Dragon",
     len(absents) == 0),
    ("Origine et licence des sources documentées",
     True),
]

bilan = pd.DataFrame(controles, columns=["controle", "statut"])
bilan["statut"] = bilan["statut"].map({True: "ok", False: "a corriger"})
print(bilan.to_string(index=False))
print()
print("Contrôles en échec :", int((bilan["statut"] == "a corriger").sum()))

                                                     controle     statut
                              Les trois sources sont chargées         ok
                   Trois formats différents (CSV, JSON, XLSX)         ok
                   Volume largement supérieur à 10 000 lignes         ok
                 Encodage identifié et appliqué explicitement         ok
                                         Séparateur identifié         ok
                   patch conservé en texte et non en flottant         ok
                      date convertie avec un format explicite         ok
         Clé gameid + teamid unique là où elle est renseignée         ok
                Deux lignes équipe par partie, sans exception         ok
                             Cible result présente et binaire         ok
                         Une seule équipe gagnante par partie a corriger
               Schémas de saison comparés avant concaténation         ok
              Dérive de remplissage mesurée saison 



Contrôles en échec : 1


## 7. Problèmes détectés et suite du travail

La phase 1 ne corrige rien. Elle repère, et elle passe la main.

| Problème détecté | Constat | Traité en |
|---|---|---|
| Dérive de remplissage, non de schéma | Les cinq fichiers ont les mêmes 165 colonnes, mais `void_grubs` ne se remplit qu'à partir de 2024 et `atakhans` de 2025. Une comparaison d'en-têtes ne voit rien | Phase 2 pour le diagnostic, phase 3 pour la décision |
| Snapshot à 15 minutes absent sur environ 11 % des lignes équipe | La règle d'inclusion exige que les **deux** lignes d'une partie soient complètes, sinon les deux sont supprimées, sans quoi l'équilibre 50/50 casse | Phase 3 |
| `datacompleteness` et disponibilité réelle ne coïncident pas | Le drapeau et le comptage des colonnes `at15` donnent des totaux légèrement différents. Le cadrage l'anticipait : on se fie aux colonnes, le drapeau reste un signal secondaire | Phase 2 |
| `teamid` manquant sur une petite part des lignes | Environ 1 à 3 % des lignes équipe selon la saison. Ce sont de vraies parties opposant deux équipes distinctes, pas des doublons : c'est `duplicated` qui assimile deux `NaN` | Phase 3, jointure sur `teamname` en secours |
| Référentiel de ligues initialement incomplet | 68 des 84 codes manquaient, dont LCKC avec 4 812 lignes. Ces lignes auraient eu `region` et `tier_ligue` à `NaN`, or ces deux colonnes sont des features. Référentiel étendu aux 84 codes, couverture vérifiée à 100 % | Corrigé en phase 1 |
| Classification de 16 % des lignes à confiance moyenne | Pour une trentaine de codes, la région est établie par les rosters mais le nom exact de la compétition reste déduit. Le tier est le champ le plus incertain | À citer en soutenance sur la question business 5 |
| Région `CEI` réduite à 32 lignes | Modalité trop rare pour être encodée telle quelle | Phase 4, regroupement des régions rares |
| 3 parties sans vainqueur | Les deux lignes portent `result = 0`. Six lignes inutilisables, à supprimer par paires | Phase 3 |
| `year` vaut 2027 sur 10 lignes | Parties de septembre 2026, patch 16.17, étiquetées saison 2027. Un découpage sur `year` les exclurait du train comme du test | Phase 3 |
| Noms de champions localisés | 5 champions portent un nom français différent du nom anglais utilisé par Oracle's Elixir | Corrigé dès la phase 1, jointure sur `en_US` |
| Majorité de colonnes post-partie | 165 colonnes, dont une large majorité d'agrégats de fin de partie qui contiennent le résultat, directement ou non | Phase 3, via `config.LEAKY_COLUMNS` |
| Les patchs débordent des saisons | Le fichier 2023 contient des patchs 12.x, le fichier 2026 des patchs 15.x. `patch` n'identifie donc pas une saison à lui seul | Phase 4, via `patch_seq` |

### Choix laissé ouvert

Ce notebook ne persiste aucun fichier. Chaque notebook ultérieur recharge donc les CSV,
ce qui coûte une à deux minutes. L'alternative est d'écrire ici un parquet des lignes
équipe brutes pour accélérer les phases suivantes.

L'option n'a pas été retenue pour ne pas brouiller le découpage annoncé dans le README,
où `data/interim/` est le livrable de la phase 3 et contient des données nettoyées, pas
brutes. Si le temps de rechargement devient gênant, c'est le premier réglage à changer.

## 8. Réflexion

**Fiabilité des sources.** Oracle's Elixir est la plus fiable : maintenue
quotidiennement, utilisée par l'ensemble de l'industrie esport, et elle porte la cible.
Data Dragon vient de Riot, donc de l'éditeur du jeu, mais elle ne décrit que l'état
**courant** du jeu, ce qui est une limite réelle détaillée ci-dessous. Le référentiel
des ligues reste le maillon faible, puisque je l'ai construit moi-même. Sa couverture
est maintenant complète et vérifiée, mais son classement en tiers reste un jugement, pas
une mesure, et environ 16 % des lignes reposent sur une identification de confiance
moyenne. C'est la limite à annoncer soi-même en soutenance plutôt qu'à se faire opposer.

**Ponts entre les sources.** `league` relie les lignes équipe au référentiel. Le nom
anglais du champion relie les lignes joueur à Data Dragon. `gameid` relie les lignes
joueur aux lignes équipe, et c'est ce pont qui permettra de reconstruire les cinq picks
d'une équipe en phase 4.

**Données que j'aurais aimé avoir.** Deux manques.

Le premier est l'historique de Data Dragon. Les tags d'un champion sont ceux du patch
courant, alors que les rôles évoluent au fil des saisons. Un champion joué en support en
2022 mais classé Marksman aujourd'hui sera mal décrit rétrospectivement. Data Dragon
expose bien les anciennes versions, mais les aligner patch par patch sur cinq saisons
dépasse le périmètre du projet. La limite est donc assumée, et sera rappelée en
soutenance plutôt que passée sous silence.

Le second est l'ordre de la draft. Oracle's Elixir donne les picks et les bans, mais ni
leur ordre ni le side du premier pick, qui portent une partie de l'information
stratégique.

**Granularité.** Les trois sources sont à des grains différents : la ligne équipe dans
une partie, le champion, la ligue. Les deux sources d'enrichissement se joignent en un
vers plusieurs sur la source principale, sans jamais dupliquer de ligne équipe, à
condition que leurs clés soient uniques. C'est le cas, et la section 4 le vérifie.